#### Converts html files to a single merged markdown file
with metadata that's supposed to be readable by NotebookLM and 

In [1]:
from icecream import ic
import pathlib as pl
from pyzotero import zotero
import sys

# refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
# #ic(refwrangle_dir)
# sys.path.append(str(refwrangle_dir))
# import refwrangle as rfw


refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw

In [2]:
zotinfo = rfw.load_pickle_data(rfw.extractedZoteroEntriesFNm) # all attachments from load_zotero_data.ipynb

# select which attachments to merge
merge_attachments = zotinfo['attachment_files'].query('contentType == "text/html"')
merge_attachments = merge_attachments[merge_attachments['parentCollections'].apply(lambda x: 'Hot Takes' in x)]

# Standardize merged output separator page meta information names
info_renames = dict(Title='parentTitle',Author='parentFirstCreator',Source='parentVenue', Date='parentDate')
metaCols = merge_attachments[info_renames.values()].rename(info_renames, axis=1)

# Do the merge
metadata_list = []
for ix, fullpath in merge_attachments.file_fullpath.items():
    metadata_list.append({'file':str(fullpath), 'metainfo':metaCols.loc[ix].to_dict()})

html_files = merge_attachments.file_fullpath

Reading from C:\Users\scott\OneDrive\share\ref\refwrangle\dat\zotero_entries.pkl...


: 

In [ ]:
from bs4 import BeautifulSoup
from readability import Document
from markdownify import markdownify as md
import chardet
import pandas as pd

# Too strict, almost totally removes article text sometimes
#
# def clean_html_readability(html):
#     soup = BeautifulSoup(html, 'html.parser')
    
#     # Remove newsletter sections, privacy notices, and similar elements
#     selectors_to_remove = [
#         '.newsletter-signup',
#         '.privacy-notice',
#         '.newsletter-promotion',
#         '[data-component="EmailSignup"]',
#         '.ad',
#         '.advertisement',
#         '.social-share',
#         '.subscription-promo'
#     ]
    
#     for selector in selectors_to_remove:
#         for element in soup.select(selector):
#             element.decompose()
            
#     # Remove elements with specific text content
#     text_patterns = [
#         'Privacy Notice:',
#         'Newsletter',
#         'Subscribe to',
#         'Sign up for'
#     ]
    
#     for pattern in text_patterns:
#         for element in soup.find_all(string=lambda text: text and pattern in text):
#             parent = element.parent
#             if parent:
#                 parent.decompose()
    
#     return str(soup)

# perplexity thinks this is less strict, yet it uses hard-coded phrases
# def clean_html_readability(html):
#     soup = BeautifulSoup(html, 'html.parser')
    
#     # Remove only specific promotional and advertising elements
#     selectors_to_remove = [
#         '.ad',
#         '.advertisement',
#         '.social-share-buttons',
#         '.newsletter-inline-signup',
#         '.popup-newsletter',
#         '.subscription-wall',
#         '[data-component="AdSlot"]',
#         '.sticky-footer-ad'
#     ]
    
#     for selector in selectors_to_remove:
#         for element in soup.select(selector):
#             element.decompose()
            
#     # Only remove elements that exactly match these phrases
#     text_patterns = [
#         'Subscribe now for full access',
#         'Sign up for our daily newsletter',
#         'Advertisement',
#         'Sponsored Content'
#     ]
    
#     for pattern in text_patterns:
#         for element in soup.find_all(string=lambda text: text and text.strip() == pattern):
#             parent = element.parent
#             if parent and len(parent.get_text(strip=True)) < 100:  # Only remove short elements
#                 parent.decompose()
    
#     return str(soup)

def clean_and_convert_to_markdown(html):
    # First clean the HTML and remove images
    soup = BeautifulSoup(html, 'html.parser')
    for img in soup.find_all('img'):
        img.decompose()
    
    cleaned_html = str(soup)
    
    # Extract main content
    doc = Document(cleaned_html)
    content = doc.summary()
    
    # Convert to markdown with specific settings
    markdown_content = md(
        content,
        heading_style="ATX",
        strip=['img']  # Additional image stripping during markdown conversion
    )
    
    return markdown_content

def write_markdown_to_file(markdown_content, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(markdown_content)

def html_to_markdown(html_file, markdown_file):
    # with open(html_file, 'rb') as f:
    #     raw_data = f.read()
    #     try:
    #         detected = chardet.detect(raw_data)
    #         html = raw_data.decode(detected['encoding'] or 'utf-8')
    #     except UnicodeDecodeError:
    #         html = raw_data.decode('iso-8859-1')

    html = rfw.read_html_file(html_file)

    markdown_content = clean_and_convert_to_markdown(html)

    # cleaned_html = clean_html_readability(html)
    
    # doc = Document(cleaned_html)
    # content = doc.summary()
    # markdown_content = md(content, heading_style="ATX")

    write_markdown_to_file(markdown_content, markdown_file)
    

source_file_ext = 'html' # 'pdf'
dest_file_ext = 'md' # 'pdf
if source_file_ext != 'html':
    raise ValueError(f'{source_file_ext=} not implemented')
if dest_file_ext == 'md':
    min_dest_bytes = 50
else:
    raise ValueError(f'{source_file_ext=} not implemented')

#convert_source_to_dest= rfw.convert_html_to_pdf_subproc
convert_source_to_dest= html_to_markdown

def add_fail(fail_list, fileInfo, failInfo):
    fail_list.append(failInfo | fileInfo)

# destSpecs = {'pdf' : dict(min_dest_bytes=100e3,
#                           fileExt='pdf',
#                           cachedir=rfw.html2pdf_cachedir}

source_file_failures = []
for ix, attachment in merge_attachments.iterrows():
    fileInfo = dict(citekey=attachment.parentCitekey)
    source_file = attachment.file_fullpath
    try:
        if source_file_ext != 'html':
            raise ValueError(f'{source_file_ext=} not implemented')
        if dest_file_ext != 'md':
            raise ValueError(f'{dest_file_ext=} not implemented')

        dest_file = rfw.processed_source_cachedir / f"{source_file.stem}.{dest_file_ext}"
    except Exception as e:
        print(f'Skipping bad {source_file_ext} file ({e}) {fileInfo["citekey"]}')
        add_fail(source_file_failures, fileInfo, dict(fail_type='bad_source_file',
                                                      fail_msg=e))
        continue
    
    if dest_file.exists():
        print(f'dest file exists, so skipping: {fileInfo["citekey"]}')
        continue
    
    fileInfo = fileInfo | dict(source_file=source_file.name, dest_file=dest_file.name)
    nbytes = dest_file.stat().st_size if dest_file.exists() else 0
    if (not dest_file.exists() or nbytes < min_dest_bytes
        or source_file.stat().st_mtime > dest_file.stat().st_mtime):
        print(f"{source_file.name} --> {dest_file.name}")
        try:
            convert_source_to_dest(source_file, dest_file)
            nbytes = dest_file.stat().st_size if dest_file.exists() else 0
            if nbytes < min_dest_bytes:
                print(f"{dest_file.name} {nbytes=} less than {min_dest_bytes} on {source_file.name}")
            add_fail(source_file_failures, fileInfo, dict(fail_type='tooshort', nbyte=nbytes))
        except Exception as e:
            print(f"Failed on {source_file.name} ({e})")
            add_fail(source_file_failures, fileInfo, dict(fail_type='exception'))
    else:
        print(f"Skipping {source_file.name}, Destination file is up to date.")


print(f"\nDone: {(nFails := len(source_file_failures))} conversion falures")
if nFails > 0:
    print("Failures:")
    display(pd.DataFrame(source_file_failures))


Cousens24voterEngageHistoryPost.html --> Cousens24voterEngageHistoryPost.md
Otte24whyVotersNoVote.html --> Otte24whyVotersNoVote.md
Brownstein24trumpBetrayRural.html --> Brownstein24trumpBetrayRural.md
Marantz24demsPartyOfElites.html --> Marantz24demsPartyOfElites.md
DavisWSUStudyPresidential.html --> DavisWSUStudyPresidential.md
Johnson24identityPoliticsIsntWhy.html --> Johnson24identityPoliticsIsntWhy.md
Blueprint24pollPosHarrisCase.html --> Blueprint24pollPosHarrisCase.md
FitzGerald24howBigTrumpWin.html --> FitzGerald24howBigTrumpWin.md
Epstein24demBrightSpots.html --> Epstein24demBrightSpots.md
Dowd24demsMistakenIdentyPol.html --> Dowd24demsMistakenIdentyPol.md
Kristof24demsNeedWrkingClssActLikeIt.html --> Kristof24demsNeedWrkingClssActLikeIt.md
McArdle24demsStopAbortCmpgn.html --> McArdle24demsStopAbortCmpgn.md
Bradner24trumpVoterShifts.html --> Bradner24trumpVoterShifts.md
Sanders24demoGroups5Voted.html --> Sanders24demoGroups5Voted.md
Stewart24divideDemsWorkingClass.html --> Ste

In [66]:
# from readability import Document
# from datetime import datetime
# import chardet

# def html_to_markdown_collection(html_files, metadata_list):
#     combined_markdown = "# Collected Web Articles\n\n"
    
#     for html_file, metadata in zip(html_files, metadata_list):
#         # Read HTML with encoding detection
#         with open(html_file, 'rb') as f:
#             raw_data = f.read()
#             detected = chardet.detect(raw_data)
#             html = raw_data.decode(detected['encoding'] or 'utf-8')
        
#         # Parse with readability
#         doc = Document(html)
#         content = doc.summary()
#         title = doc.title()
        
#         # Create article separator and metadata section
#         combined_markdown += "\n---\n## Article: " + title + "\n\n"
        
#         # Add metadata block in YAML-like format
#         combined_markdown += "```"
#         for key, value in metadata.items():
#             combined_markdown += f"{key}: {value}\n"
#         combined_markdown += "```\n\n"
        
#         # Add content with clear boundaries
#         combined_markdown += "### Content\n\n"
#         combined_markdown += content + "\n\n"
        
#         # Add explicit end marker
#         combined_markdown += "### End of Article\n\n"
    
#     return combined_markdown

# def write_markdown_to_file(markdown_content, filename):
#     with open(filename, 'w', encoding='utf-8') as f:
#         f.write(markdown_content)


In [67]:
# works but spits out a lot of html, is very slow (> 20 mins), and seems to be getting article headline hierarchy.  frontmatter is also just a dump of dict, rather than frontmatter formatted.
# from readability import Document
# import chardet

# def html_to_markdown_collection(html_files, metadata_list):
#     combined_markdown = "# Collected Web Articles\n\n"
    
#     i = 0
#     for html_file, metadata in zip(html_files, metadata_list):
#         # Read HTML with encoding detection and fallback
#         i += 1
#         if i> 10:
#             print("early stop")
#             break
#         ic(i, html_file)
#         with open(html_file, 'rb') as f:
#             raw_data = f.read()
#             try:
#                 detected = chardet.detect(raw_data)
#                 html = raw_data.decode(detected['encoding'] or 'utf-8')
#             except UnicodeDecodeError:
#                 html = raw_data.decode('iso-8859-1')
        
#         # Parse with readability
#         doc = Document(html)
#         content = doc.summary()
#         title = doc.title()
        
#         # Create article separator and metadata section
#         combined_markdown += f"\n---\n## Article: {title}\n\n"
        
#         # Add metadata block in YAML-like format
#         combined_markdown += "```"
#         for key, value in metadata.items():
#             combined_markdown += f"{key}: {value}\n"
#         combined_markdown += "```\n\n"
        
#         # Add content with clear boundaries
#         combined_markdown += "### Content\n\n"
#         combined_markdown += content + "\n\n"
        
#         # Add explicit end marker
#         combined_markdown += "### End of Article\n\n"
    
#     return combined_markdown

# def write_markdown_to_file(markdown_content, filename):
#     with open(filename, 'w', encoding='utf-8') as f:
#         f.write(markdown_content)

In [68]:
# # Works but doesn't delete as much junk as obsidian web clipper plugin

# from readability import Document
# from markdownify import markdownify as md
# import chardet

# def html_to_markdown_collection(html_files, metadata_list):
#     combined_markdown = "# Collected Web Articles\n\n"
    
#     i=0
#     for html_file, metadata in zip(html_files, metadata_list):
#         i += 1
#         if i>10:
#             print('exiting early')
#             break
#         ic(i, html_file)               

#         # Read HTML with encoding detection and fallback
#         with open(html_file, 'rb') as f:
#             raw_data = f.read()
#             try:
#                 detected = chardet.detect(raw_data)
#                 html = raw_data.decode(detected['encoding'] or 'utf-8')
#             except UnicodeDecodeError:
#                 html = raw_data.decode('iso-8859-1')
        
#         # Parse with readability
#         doc = Document(html)
#         content = doc.summary()
#         title = doc.title()
        
#         # Convert HTML to Markdown
#         markdown_content = md(content, heading_style="ATX")
        
#         # Create article separator and metadata section
#         combined_markdown += f"\n---\n## Article: {title}\n\n"
        
#         # Add metadata block in YAML-like format
#         combined_markdown += "```"
#         for key, value in metadata.items():
#             combined_markdown += f"{key}: {value}\n"
#         combined_markdown += "```\n\n"
        
#         # Add content with clear boundaries
#         combined_markdown += "### Content\n\n"
#         combined_markdown += markdown_content + "\n\n"
        
#         # Add explicit end marker
#         combined_markdown += "### End of Article\n\n"
    
#     return combined_markdown

# def write_markdown_to_file(markdown_content, filename):
#     with open(filename, 'w', encoding='utf-8') as f:
#         f.write(markdown_content)

In [70]:

# def html_to_markdown_collection(html_files, metadata_list):
#     combined_markdown = "# Collected Web Articles\n\n"
    
#     i=0    
#     for html_file, metadata in zip(html_files, metadata_list):

#         i += 1
#         if i>10:
#             print("stopping early")
#             break
#         ic(i, html_file)

#         with open(html_file, 'rb') as f:
#             raw_data = f.read()
#             try:
#                 detected = chardet.detect(raw_data)
#                 html = raw_data.decode(detected['encoding'] or 'utf-8')
#             except UnicodeDecodeError:
#                 html = raw_data.decode('iso-8859-1')
        
#         # Clean HTML before passing to Readability
#         cleaned_html = clean_html_readability(html)
        
#         # Initialize Readability
#         doc = Document(cleaned_html)
        
#         content = doc.summary()
#         title = doc.title()
        
#         # Convert to markdown
#         markdown_content = md(content, heading_style="ATX")
        
#         # Create article separator and metadata section
#         combined_markdown += f"\n---\n## Article: {title}\n\n"
        
#         # Add metadata block in YAML-like format
#         combined_markdown += "```"
#         for key, value in metadata.items():
#             combined_markdown += f"{key}: {value}\n"
#         combined_markdown += "```\n\n"
        
#         # Add content with clear boundaries
#         combined_markdown += "### Content\n\n"
#         combined_markdown += markdown_content + "\n\n"
        
#         # Add explicit end marker
#         combined_markdown += "### End of Article\n\n"
    
#     return combined_markdown


In [71]:
# markdown_content = html_to_markdown_collection(html_files, metadata_list)

# import pathlib as pl
# outFNm = pl.Path('~/ref/obsidian/Obsidian Share Vault/combined_htmls.md').expanduser()
# print(f'writing to {str(outFNm)}')
# write_markdown_to_file(markdown_content, outFNm)